In [ ]:
import pandas as pd
import re
from difflib import SequenceMatcher

# =====================================================
# 1. INPUT FILES
# =====================================================
ieee_file = "screening_results.xlsx"
scopus_file = "screening_results_scopus.xlsx"

# =====================================================
# 2. OUTPUT FILES
# =====================================================
merged_all_file = "IEEE_Scopus_all_with_duplicates.xlsx"
merged_clean_file = "IEEE_Scopus_all_clean_no_duplicates.xlsx"

# =====================================================
# 3. LOAD EXCEL FILES
# =====================================================
df_ieee = pd.read_excel(ieee_file, engine="openpyxl")
df_scopus = pd.read_excel(scopus_file, engine="openpyxl")

# Add source column
df_ieee["Source"] = "IEEE"
df_scopus["Source"] = "Scopus"

# Merge both files
df = pd.concat([df_ieee, df_scopus], ignore_index=True)

print("IEEE records:", len(df_ieee))
print("Scopus records:", len(df_scopus))
print("Total merged records:", len(df))

# =====================================================
# 4. NORMALIZATION FUNCTIONS
# =====================================================
def normalize_title(title):
    if pd.isna(title):
        return ""
    title = str(title).lower()
    title = re.sub(r"[^\w\s]", " ", title)
    title = re.sub(r"\s+", " ", title).strip()
    return title

def normalize_doi(doi):
    if pd.isna(doi):
        return ""
    doi = str(doi).lower().strip()
    doi = doi.replace("https://doi.org/", "")
    doi = doi.replace("http://doi.org/", "")
    doi = doi.replace("doi:", "")
    doi = doi.strip()
    return doi

# =====================================================
# 5. CHECK REQUIRED COLUMNS
# =====================================================
if "title" not in df.columns:
    raise ValueError("Column 'title' not found. Please check your Excel files.")

if "doi" not in df.columns:
    print("Warning: column 'doi' not found. Duplicate detection will use title only.")
    df["doi"] = ""

# =====================================================
# 6. CREATE NORMALIZED COLUMNS
# =====================================================
df["title_norm"] = df["title"].apply(normalize_title)
df["doi_norm"] = df["doi"].apply(normalize_doi)

# =====================================================
# 7. DUPLICATE DETECTION
# =====================================================
df["isDuplicate"] = False
df["DuplicateReason"] = ""
df["DuplicateOfIndex"] = ""

seen_doi = {}
seen_title = {}

for idx, row in df.iterrows():
    doi = row["doi_norm"]
    title = row["title_norm"]

    # Duplicate by DOI
    if doi != "":
        if doi in seen_doi:
            df.at[idx, "isDuplicate"] = True
            df.at[idx, "DuplicateReason"] = "Same DOI"
            df.at[idx, "DuplicateOfIndex"] = seen_doi[doi]
            continue
        else:
            seen_doi[doi] = idx

    # Duplicate by exact normalized title
    if title != "":
        if title in seen_title:
            df.at[idx, "isDuplicate"] = True
            df.at[idx, "DuplicateReason"] = "Same normalized title"
            df.at[idx, "DuplicateOfIndex"] = seen_title[title]
            continue
        else:
            seen_title[title] = idx

# =====================================================
# 8. OPTIONAL FUZZY TITLE MATCHING
# =====================================================
# This detects titles that are almost identical but not exactly the same.
# Example: punctuation differences, small spelling differences.

FUZZY_MATCHING = True
TITLE_SIMILARITY_THRESHOLD = 0.95

if FUZZY_MATCHING:
    for i in range(len(df)):
        if df.at[i, "isDuplicate"]:
            continue

        title_i = df.at[i, "title_norm"]

        if title_i == "":
            continue

        for j in range(i):
            if df.at[j, "isDuplicate"]:
                continue

            title_j = df.at[j, "title_norm"]

            if title_j == "":
                continue

            similarity = SequenceMatcher(None, title_i, title_j).ratio()

            if similarity >= TITLE_SIMILARITY_THRESHOLD:
                df.at[i, "isDuplicate"] = True
                df.at[i, "DuplicateReason"] = f"Similar title ({similarity:.2f})"
                df.at[i, "DuplicateOfIndex"] = j
                break

# =====================================================
# 9. CLEAN DATASET WITHOUT DUPLICATES
# =====================================================
df_clean = df[df["isDuplicate"] == False].copy()

# =====================================================
# 10. REMOVE HELPER COLUMNS BEFORE EXPORT
# =====================================================
df_export = df.drop(columns=["title_norm", "doi_norm"])
df_clean_export = df_clean.drop(columns=["title_norm", "doi_norm"])

# =====================================================
# 11. SAVE FILES
# =====================================================
df_export.to_excel(merged_all_file, index=False)
df_clean_export.to_excel(merged_clean_file, index=False)

# =====================================================
# 12. SUMMARY
# =====================================================
print("\n===== SUMMARY =====")
print("Total IEEE records:", len(df_ieee))
print("Total Scopus records:", len(df_scopus))
print("Total merged records:", len(df))
print("Duplicates detected:", df["isDuplicate"].sum())
print("Clean records:", len(df_clean))

# =====================================================
# 13. COUNT DECISIONS (Include / Exclude / Uncertain)
# =====================================================
if "decision" in df_clean.columns:
    include_count = (df_clean["decision"].str.lower() == "include").sum()
    exclude_count = (df_clean["decision"].str.lower() == "exclude").sum()
    uncertain_count = (df_clean["decision"].str.lower().isin(["maybe", "uncertain"])).sum()

    print("\n===== DECISION COUNTS =====")
    print("Include:", include_count)
    print("Exclude:", exclude_count)
    print("Uncertain:", uncertain_count)
else:
    print("\nWarning: 'Decision' column not found in dataset.")

print("\nFiles created:")
print("-", merged_all_file)
print("-", merged_clean_file)

IEEE records: 189
Scopus records: 372
Total merged records: 561

===== SUMMARY =====
Total IEEE records: 189
Total Scopus records: 372
Total merged records: 561
Duplicates detected: 98
Clean records: 463

===== DECISION COUNTS =====
Include: 172
Exclude: 186
Uncertain: 105

Files created:
- IEEE_Scopus_all_with_duplicates.xlsx
- IEEE_Scopus_all_clean_no_duplicates.xlsx
